In [51]:
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from typing import TypedDict, Literal
from pydantic import BaseModel, Field
from dotenv import load_dotenv
load_dotenv()

True

In [52]:
llm = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-120b",
    task="text-generation"
)

model = ChatHuggingFace(llm=llm)

In [53]:
class SentimentSchema(BaseModel):
    sentiment: Literal["positive", "negative"] = Field(
        description="The sentiment of the review"
    )

class ReviewState(TypedDict):
    review: str
    sentiment: Literal["positive", "negative"]
    diagnosis: dict
    response: str

class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description="The type of issue identified in the review")
    tone: Literal["Angry", "Frustrated", "Neutral"] = Field(description="The tone of the review")
    urgency: Literal["Low", "Medium", "High"] = Field(description="The urgency of the issue identified in the review")

In [54]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate

# Use JsonOutputParser instead of with_structured_output
parser = JsonOutputParser(pydantic_object=SentimentSchema)
parser2 = JsonOutputParser(pydantic_object=DiagnosisSchema)

# Create a prompt that instructs the model to output JSON
prompt = PromptTemplate(
    template="Analyze the sentiment of the following review and respond in JSON format with 'sentiment' field.\n\nReview: {review}\n\n{format_instructions}",
    input_variables=["review"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)
prompt2 = PromptTemplate(
    template="Based on the review and its sentiment, diagnose the issue and respond in JSON format with 'issue_type', 'tone', and 'urgency' fields.\n\nReview: {review}\nSentiment: {sentiment}\n\n{format_instructions}",
    input_variables=["review", "sentiment"],
    partial_variables={"format_instructions": parser2.get_format_instructions()}
)

# Chain the model with the parser
structured_model = prompt | model | parser
structured_model2 = prompt2 | model | parser2

In [55]:
structured_model.invoke({"review": "This is great!"})['sentiment']
structured_model2.invoke({"review": "I hate this product", "sentiment": 'negative'})

{'issue_type': 'Other', 'tone': 'Angry', 'urgency': 'High'}

In [56]:
def find_sentiment(state: ReviewState):
    sentiment = structured_model.invoke({"review": state['review']})['sentiment']
    return {"sentiment": sentiment}

def check_sentiment(state: ReviewState) -> Literal['positive_response', 'run_diagnosis']:
    if state['sentiment'] == 'positive':
        return 'positive_response'
    else:
        return 'run_diagnosis'

def positive_response(state: ReviewState):
    prompt = f"""Write a response to the following positive review:\n\n{state['review']}"""
    response = model.invoke(prompt).content
    return {'response': response}

def run_diagnosis(state: ReviewState):
    prompt = f"""Diagnose the issue based on the following review and its sentiment:\n\nReview: {state['review']}\nSentiment: {state['sentiment']}"""
    response = structured_model2.invoke({"review": state['review'], "sentiment": state['sentiment']})
    return {'diagnosis': response}

def negative_response(state: ReviewState):
    prompt = f"""Write a response to the following negative review, addressing the identified issue:\n\nReview: {state['review']}\nSentiment: {state['sentiment']}\nDiagnosis: {state['diagnosis']}"""
    response = model.invoke(prompt).content
    return {'response': response}

In [57]:
graph = StateGraph(ReviewState)

graph.add_node('find_sentiment', find_sentiment)
graph.add_node('positive_response', positive_response)
graph.add_node('run_diagnosis', run_diagnosis)
graph.add_node('negative_response', negative_response)


graph.add_edge(START, 'find_sentiment')
graph.add_conditional_edges('find_sentiment', check_sentiment)
graph.add_edge('positive_response', END)
graph.add_edge('run_diagnosis', 'negative_response')
graph.add_edge('negative_response', END)

workflow = graph.compile()

In [58]:
initial_state = {
    'review': """The Boat Airdopes 141 fail to live up to expectations despite their popularity. The sound quality is heavily bass-focused, which often overpowers vocals and makes music sound unbalanced. If you prefer clear and detailed audio, these earbuds can be quite disappointing.

The build quality also feels cheap and not very durable. The plastic case and earbuds seem fragile, and the lid can become loose after some use. Additionally, the touch controls are inconsistent and sometimes don’t respond properly, which becomes frustrating during daily use.

Battery life, while advertised as long-lasting, doesn’t always match real-world performance, especially with heavy usage. Another major drawback is the poor microphone quality—calls can sound muffled, particularly in noisy environments."""
}

workflow.invoke(initial_state)

{'review': 'The Boat Airdopes 141 fail to live up to expectations despite their popularity. The sound quality is heavily bass-focused, which often overpowers vocals and makes music sound unbalanced. If you prefer clear and detailed audio, these earbuds can be quite disappointing.\n\nThe build quality also feels cheap and not very durable. The plastic case and earbuds seem fragile, and the lid can become loose after some use. Additionally, the touch controls are inconsistent and sometimes don’t respond properly, which becomes frustrating during daily use.\n\nBattery life, while advertised as long-lasting, doesn’t always match real-world performance, especially with heavy usage. Another major drawback is the poor microphone quality—calls can sound muffled, particularly in noisy environments.',
 'sentiment': 'negative',
 'diagnosis': {'issue_type': 'Performance',
  'tone': 'Frustrated',
  'urgency': 'Medium'},
 'response': 'Dear\u202f[Customer Name],\n\nThank you for taking the time to sh